# Collected edits

All three dermatologists go into one codebook per model. Reliability, text generality, and image generality then run on that full list, in that order. A query can retrieve another user's fact.

The overlap list is printed first. Those cases share an image and the original question, so their keys sit on top of each other.

`dermedit` kernel. Set `OVERWRITE = True` to recompute saved runs.

In [1]:
import sys
from pathlib import Path

PERTURB = Path("/scratch/jq2uw/MME/derm_vlm_edit/editing/perturb_generality")
sys.path.insert(0, str(PERTURB))
from utils.bootstrap import bootstrap
bootstrap()

import config as C

USERS = [
    "jesse.y.liu.md@gmail.com",
    "achiou@stanford.edu",
    "lhunterellul",
]
MODELS = ["medgemma", "dermatollama"]
RATIONALE_MODE = C.RATIONALE_MODE
NORM_ANSWER = "y16"
SUBSAMPLE = 0
MAX_NEW_TOKENS = C.MAX_NEW_TOKENS
SEED = 42
OVERWRITE = False

EMBED_MODE = "standalone_dual"
IMAGE_ENCODER = "dinov2"
IMAGE_POOL = "cls"
TEXT_ENCODER = "sbert"
SCALER_DATASET = "coco"

OUT = PERTURB / "results" / "collect"
POOL = "all"
print("results:", OUT)

results: /scratch/jq2uw/MME/derm_vlm_edit/editing/perturb_generality/results/collect


In [ ]:
import gc

from IPython.display import Markdown, display

from utils.edit import edit
from utils.edits import collect_all, overlap_report
from utils.eval import load_suite, load_vlm, render_table, run_suite, suite_complete

rows = []
for model_short in MODELS:
    if suite_complete(OUT, POOL, model_short) and not OVERWRITE:
        print(f"{POOL} / {model_short}: loaded")
        rows.append(load_suite(OUT, POOL, model_short))
        continue

    edits, _ = collect_all(
        USERS, model_short, rationale_mode=RATIONALE_MODE,
        norm_answer=NORM_ANSWER, subsample=SUBSAMPLE,
    )
    overlaps = overlap_report(edits)
    print(f"{model_short}: {len(edits)} edits, {len(overlaps)} cases shared by more than one user")
    for row in overlaps:
        agree = "agree" if row["answers_agree"] else "differ"
        print(f"  {row['case_id']}  {agree}: {', '.join(row['answers'])}")
    if not edits:
        print(f"skip {model_short}")
        continue

    import torch
    vlm, config = load_vlm(
        model_short, max_new_tokens=MAX_NEW_TOKENS, seed=SEED,
        embed_mode=EMBED_MODE, image_encoder=IMAGE_ENCODER,
        image_pool=IMAGE_POOL, text_encoder=TEXT_ENCODER,
        scaler_dataset=SCALER_DATASET,
    )
    editor, queries = edit(vlm, config, edits)
    rows.append(run_suite(
        editor, vlm, queries, user=POOL, model_short=model_short,
        out_dir=OUT, seed=SEED, norm_answer=NORM_ANSWER,
        max_new_tokens=MAX_NEW_TOKENS,
    ))
    del vlm, editor
    gc.collect()
    torch.cuda.empty_cache()

display(Markdown(render_table(rows)))

all / medgemma: loaded
dermatollama: 33 edits, 2 cases shared by more than one user
  1025_combined  differ: actinic keratosis, lichenoid keratosis
  845_combined  agree: actinic keratosis
